# Clustering conformations

FleXgeo2 can cluster ensemble conformations with HDBSCAN. Per-residue clustering groups models using one residue's `(curvature, torsion)` values. Residue-range clustering builds one combined geometric signature across a selected residue window.

In [ ]:
from pathlib import Path
import sys
import tempfile


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "pdb2lj5.pdb").is_file():
            return candidate
    raise FileNotFoundError("Could not find the FleXgeo2 repo root containing pyproject.toml and pdb2lj5.pdb")


REPO_ROOT = find_repo_root()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

try:
    import hdbscan  # noqa: F401
    import matplotlib.image as mpimg
    import matplotlib.pyplot as plt
    import melodia_py  # noqa: F401
    import pandas as pd
    from flexgeo2 import AnalysisConfig, ClusteringConfig, FlexGeo2App, OutputConfig
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        f"Missing dependency: {exc.name}. Install the project in your notebook kernel with `pip install -e .` "
        "or, if you use uv, run `uv sync` and select the project environment as the Jupyter kernel."
    ) from exc

PDB_FILE = REPO_ROOT / "pdb2lj5.pdb"
tmpdir = tempfile.TemporaryDirectory(prefix="flexgeo2_clustering_")
OUTPUT_DIR = Path(tmpdir.name)
print(f"Temporary output directory: {OUTPUT_DIR}")

## Run residue and residue-range clustering

This example clusters each residue independently and also clusters the window `45-54` as one combined signature. HDBSCAN labels `-1` mean noise or unassigned conformations.

In [ ]:
result = FlexGeo2App().run(
    AnalysisConfig(
        pdb_file=PDB_FILE,
        chains=["A"],
        n_jobs=1,
        clustering=ClusteringConfig(
            cluster_residues=True,
            cluster_residue_ranges=["45-54"],
            min_cluster_size=5,
        ),
        output=OutputConfig(output_dir=OUTPUT_DIR, verbose=True, write_files=True),
    )
)

## Per-residue clustering summary

`n_clusters` counts non-noise HDBSCAN clusters for each residue. `noise_fraction` is the fraction of models labeled `-1`.

In [ ]:
residue_clusters = result.residue_clustering
assert residue_clusters is not None
residue_clusters.summary_df.sort_values(["n_clusters", "noise_fraction"], ascending=[False, True]).head(15)

In [ ]:
residue_clusters.assignments_df.head()

## Show a selected residue cluster plot

FleXgeo2 writes one plot per residue. Instead of displaying all of them, choose a residue with the most non-noise clusters and display that one.

In [ ]:
selected_residue = residue_clusters.summary_df.sort_values(
    ["n_clusters", "noise_fraction"], ascending=[False, True]
).iloc[0]
plot_path = result.outputs.cluster_plots_dir / f"A_{selected_residue['residue_label']}_clusters.png"
print(plot_path)
assert plot_path.is_file()
img = mpimg.imread(plot_path)
plt.figure(figsize=(6, 5))
plt.imshow(img)
plt.axis("off")
plt.show()

## Residue-range clustering

Residue-range clustering creates one feature vector per model by concatenating curvature and torsion values across the selected range. The `pc1` and `pc2` columns are a PCA projection for plotting the high-dimensional range signature.

In [ ]:
range_clusters = result.residue_range_clustering
assert range_clusters is not None
range_clusters.summary_df

In [ ]:
range_clusters.assignments_df.head()

In [ ]:
range_plot = result.outputs.range_cluster_plots_dir / "A_45-54_clusters.png"
assert range_plot.is_file()
img = mpimg.imread(range_plot)
plt.figure(figsize=(6, 5))
plt.imshow(img)
plt.axis("off")
plt.show()